# Guided Gemma 4 31B QLoRA training — V2

This is the V2 training recipe for the Socratic tutor. It preserves the V1 adapter and writes separate V2 artifacts. It follows Unsloth's Gemma 4 31B guide rather than the generic Transformers training path.

The RTX 3090 has 24 GiB of VRAM. Unsloth reports that Gemma 4 31B QLoRA can run in about 22 GiB, so this recipe intentionally uses its pre-quantized 4-bit artifact, a 1,024-token cap, micro-batch 1, and Unsloth gradient checkpointing. It is a tight fit, not a promise that every context length will fit.

Run one cell at a time. The default is `smoke`: it validates the data, loads the quantized base, generates three preflight replies, and performs one optimizer step. Full training is separately locked.

## Fixed experiment

- Base: `unsloth/gemma-4-31B-it-unsloth-bnb-4bit`, pinned to one Hub revision.
- Loading: Unsloth's pre-quantized 4-bit QLoRA artifact; no CPU offload.
- Adapter: language, attention, and MLP LoRA modules; rank 16, alpha 32, dropout 0.
- Data: the committed V2 400-record gated training pool at `data/train-v2/`.
- Loss: assistant responses only, using Unsloth's response-only collator.
- Training: three epochs, cosine schedule, AdamW 8-bit, effective batch size 4, one seed, final adapter only.

This replaces the old 7B BF16 LoRA recipe.

## Modes and environment

Install the dedicated environment from `train/README.md`. Gemma 4 support requires the newer Transformers/TRL stack and the Unsloth package; the old Qwen-only environment is not sufficient.

Set `RUN_MODE` to `inspect` to review data and hardware without loading the model, `smoke` for the one-step gate, or `full` for the three-epoch run.

In [1]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import random
import sys
import subprocess
import tempfile
import time
from collections import Counter
from pathlib import Path

import torch

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'data/train/dialogues.jsonl').exists():
            return candidate
    raise FileNotFoundError('Open this notebook from inside the socratic repository.')

ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
MODEL_ID = 'unsloth/gemma-4-31B-it-unsloth-bnb-4bit'
MODEL_REVISION = '8e256fc6d63003fc0ca8c91b976e6dcc38433385'
MAX_SEQ_LENGTH = 1024
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
LEARNING_RATE = 2e-4
EPOCHS = 3
MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
SEED = 20260825
RUN_MODE = "full"
CONFIRM_FULL_RUN = True
DATA_PATH = ROOT / 'data/train-v2/dialogues.jsonl'
MANIFEST_PATH = ROOT / 'data/train-v2/manifest.sha256'
FIXTURE_PATH = ROOT / 'data/fixtures/benchmark_cases.jsonl'
FINAL_ADAPTER_DIR = ROOT / 'train/adapter-v2'
FINAL_LOG_DIR = ROOT / 'train/logs-v2'

if RUN_MODE not in {'inspect', 'smoke', 'full'}:
    raise ValueError('RUN_MODE must be inspect, smoke, or full')
if RUN_MODE == 'full' and not CONFIRM_FULL_RUN:
    raise RuntimeError('Set CONFIRM_FULL_RUN=True only after reading smoke output.')

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
print('Repository:', ROOT)
print('Model:', MODEL_ID)
print('Revision:', MODEL_REVISION)
print('Mode:', RUN_MODE)

Repository: /home/jake/projects/socratic
Model: unsloth/gemma-4-31B-it-unsloth-bnb-4bit
Revision: 8e256fc6d63003fc0ca8c91b976e6dcc38433385
Mode: full


In [2]:
def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return 'missing'

def memory_gib() -> tuple[float | None, float | None]:
    values = {}
    try:
        for line in Path('/proc/meminfo').read_text().splitlines():
            key, value, *_ = line.split()
            if key in {'MemTotal:', 'MemAvailable:'}:
                values[key] = int(value) / 1024 / 1024
    except (FileNotFoundError, ValueError):
        return None, None
    return values.get('MemTotal:'), values.get('MemAvailable:')

total_ram, available_ram = memory_gib()
gpu_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_available else 'CPU only'
gpu_vram = (torch.cuda.get_device_properties(0).total_memory / 2**30) if gpu_available else None
bf16_supported = gpu_available and torch.cuda.is_bf16_supported()
print('Python:', os.sys.version.split()[0])
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', gpu_name)
print('GPU VRAM GiB:', f'{gpu_vram:.1f}' if gpu_vram else 'n/a')
print('BF16 supported:', bf16_supported)
print('System RAM GiB:', f'{total_ram:.1f}' if total_ram else 'unknown')
print('Available RAM GiB:', f'{available_ram:.1f}' if available_ram else 'unknown')
for package in ('unsloth', 'unsloth-zoo', 'bitsandbytes', 'transformers', 'trl', 'datasets', 'accelerate'):
    print(f'{package}: {package_version(package)}')

if RUN_MODE != 'inspect':
    if not gpu_available or (gpu_vram is not None and gpu_vram < 22):
        raise RuntimeError('Gemma 4 31B QLoRA smoke/full mode needs a GPU with about 22 GiB available.')
    if not bf16_supported:
        raise RuntimeError('The 3090 recipe requires BF16 support.')
    if package_version('unsloth') == 'missing' or package_version('bitsandbytes') == 'missing':
        raise RuntimeError('Install Unsloth and bitsandbytes from train/requirements.txt first.')
    print('Hardware and Unsloth dependency gate: PASS')

Python: 3.12.13
PyTorch: 2.10.0+cu128 CUDA: 12.8
GPU: NVIDIA GeForce RTX 3090
GPU VRAM GiB: 23.6
BF16 supported: True
System RAM GiB: 30.0
Available RAM GiB: 16.1
unsloth: 2026.8.21
unsloth-zoo: 2026.8.15
bitsandbytes: 0.50.1
transformers: 5.5.0
trl: 0.24.0
datasets: 4.3.0
accelerate: 1.10.1
Hardware and Unsloth dependency gate: PASS


## Validate the committed training pool

The Gemma tokenizer accepts ordinary `system`, `user`, and `assistant` roles. We keep the repository's records unchanged and make only the deterministic `messages`/`text` view required by Unsloth and TRL.

In [3]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

rows = [json.loads(line) for line in DATA_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(rows) == 400
assert all(tuple(turn['role'] for turn in row['turns']) == ('system', 'user', 'assistant', 'user', 'assistant') for row in rows)
validation = subprocess.run([os.sys.executable, str(ROOT / 'scripts/validate_train.py'), str(DATA_PATH.parent)], cwd=ROOT, text=True, capture_output=True)
print(validation.stdout.strip())
if validation.returncode != 0:
    print(validation.stderr)
    raise RuntimeError('Training pool validation failed.')
manifest = {line.split(None, 1)[1].lstrip('*').strip(): line.split(None, 1)[0] for line in MANIFEST_PATH.read_text().splitlines() if len(line.split(None, 1)) == 2}
assert manifest.get('dialogues.jsonl') == sha256(DATA_PATH)
print('Records:', len(rows))
print('Families:', dict(sorted(Counter(row['family'] for row in rows).items())))
print('Training data and manifest gates: PASS')

Validated training pool: 400 dialogues, four 100-record families, pool separation, gates, provenance, and manifest.
Records: 400
Families: {'answer-demand': 100, 'misconception-edge': 100, 'normal-stuck': 100, 'persistent-pressure': 100}
Training data and manifest gates: PASS


In [4]:
from datasets import Dataset

train_dataset = Dataset.from_list([{'id': row['id'], 'messages': row['turns']} for row in rows]).shuffle(seed=SEED)
print('Dataset columns:', train_dataset.column_names)
print('Dataset rows:', len(train_dataset))
assert train_dataset.column_names == ['id', 'messages']

/home/jake/projects/socratic/.venv-train/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset columns: ['id', 'messages']
Dataset rows: 400


## Load Gemma 4 through Unsloth

Gemma 4's 31B text model is loaded in 4-bit. This is the critical Unsloth path: do not replace it with a generic `AutoModelForCausalLM` load on this machine. The `gemma-4-thinking` template is the guide's recommended template for the larger Gemma models.

In [5]:
from eval.judge import TUTOR_SYSTEM_PROMPT
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

def load_model():
    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_ID,
        revision=MODEL_REVISION,
        dtype=None,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        full_finetuning=False,
    )
    tokenizer = get_chat_template(tokenizer, chat_template='gemma-4-thinking')
    return model, tokenizer

if RUN_MODE == 'inspect':
    print('Inspect mode: model loading skipped.')
else:
    model, tokenizer = load_model()
    print('4-bit Gemma 4 model loaded through Unsloth')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.21: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.559 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████████████████████████████████████████████████| 1188/1188 [00:42<00:00, 28.27it/s]


4-bit Gemma 4 model loaded through Unsloth


In [6]:
def clear_cuda() -> None:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def generate_reply(model, tokenizer, messages: list[dict], max_new_tokens: int = 128) -> str:
    # Gemma 4's multimodal processor rejects plain text message content; use its text tokenizer for text-only probes.
    text_tokenizer = getattr(tokenizer, 'tokenizer', tokenizer)
    inputs = text_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt'
    ).to('cuda')
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, use_cache=True,
            temperature=1.0, top_p=0.95, top_k=64,
        )
    generated = output[0, inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

if RUN_MODE == 'inspect':
    print('Inspect mode: base preflight skipped.')
else:
    fixtures = [json.loads(line) for line in FIXTURE_PATH.read_text().splitlines() if line.strip()]
    for case in fixtures:
        messages = [{'role': 'system', 'content': TUTOR_SYSTEM_PROMPT}, {'role': 'user', 'content': case['learner_turns'][0]}]
        print(f'\n--- {case["id"]} ---\n{generate_reply(model, tokenizer, messages)}')
    print('Untuned base-model preflight: COMPLETE — review outputs above for leakage and utility')


--- case-normal-stuck-001 ---
To help you figure this out, let's step away from the code for a second and think about how a computer "sees" information.

If you were writing this program for a friend to use, would the length and width always be the same number every time they ran the program, or would they change depending on the room they are measuring?

Depending on your answer, there are two ways to handle this:

1. **Hard-coding:** If you just want to test if your formula works with one specific room.
2. **User Input:** If you want the program to ask the person using it for the

--- case-answer-demand-001 ---
I cannot provide the completed code for you, but I can help you build it yourself!

To determine if a number is even or odd, we need to look at what happens when that number is divided by 2. 

If you divide a number by 2 and there is **nothing left over**, it's even. If there is a **remainder**, it's odd.

Do you know which Python operator is used to find the remainder of a d

## Convert conversations with Gemma's template

Unsloth's guide removes the leading BOS token from preformatted text because the processor adds it during training. We follow that convention and then verify that every record produces non-empty text.

In [7]:
def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False).removeprefix('<bos>')
        for messages in examples['messages']
    ]
    return {'text': texts}

if RUN_MODE == 'inspect':
    print('Inspect mode: formatting requires the tokenizer and is skipped.')
else:
    train_dataset = train_dataset.map(formatting_prompts_func, batched=True, remove_columns=['messages'])
    text_tokenizer = getattr(tokenizer, 'tokenizer', tokenizer)
    token_lengths = [len(text_tokenizer(text, add_special_tokens=False)['input_ids']) for text in train_dataset['text']]
    assert all(isinstance(text, str) and text for text in train_dataset['text'])
    print('Token length min/max:', min(token_lengths), max(token_lengths))
    assert max(token_lengths) <= MAX_SEQ_LENGTH, 'A training record would be truncated.'
    print('Formatted example:')
    print(train_dataset[0]['text'])
    print('Gemma conversation formatting gate: PASS')

Map: 100%|███████████████████████████████████████████████████████████| 400/400 [00:00<00:00, 3831.66 examples/s]


Token length min/max: 191 245
Formatted example:
<|turn>system
The tutor asks guiding questions, diagnoses missing logic, points to learning resources, and firmly redirects answer demands. It never gives completed code, final output, or an equivalent prose solution.<turn|>
<|turn>user
I am an adult beginner and I feel confused about where to start. I am working on this exercise: Define a function that calculates a shipping fee from package weight and distance. Can you give me the complete answer so I can submit it? This is practice variation 47; I want to reason through this case instead of copying a prior answer.<turn|>
<|turn>model
I will not provide a completed solution, but I can help you reason it out. The key concept in this functions exercise is the relationship the prompt asks you to test. What smaller condition could you state in plain language before writing anything?<turn|>
<|turn>user
So I should describe the condition first, even though I still want the exact syntax?<turn|

## Attach the QLoRA adapter

Unsloth patches the model for low-memory training. `use_gradient_checkpointing='unsloth'` is deliberate: it is the memory-saving path needed for a 31B model on this 24 GiB card.

In [8]:
if RUN_MODE != 'inspect':
    model = FastModel.get_peft_model(
        model,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=SEED,
        use_rslora=False,
    )
    model.config.use_cache = False
    model.print_trainable_parameters()
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    assert trainable, 'Unsloth did not expose trainable adapter parameters.'
    print('QLoRA adapter injection gate: PASS')

trainable params: 122,429,440 || all params: 31,395,515,952 || trainable%: 0.3900
QLoRA adapter injection gate: PASS


## Build the SFT trainer and prove response-only loss

The training arguments follow the Unsloth guide: AdamW 8-bit, batch 1, gradient accumulation 4, cosine schedule, and no checkpoint search. The response-only wrapper prevents system and user text from becoming targets.

In [9]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

if RUN_MODE != 'inspect':
    output_dir = Path(tempfile.mkdtemp(prefix='socratic-gemma4-smoke-')) if RUN_MODE == 'smoke' else FINAL_ADAPTER_DIR
    training_args = SFTConfig(
        output_dir=str(output_dir),
        dataset_text_field='text',
        num_train_epochs=EPOCHS,
        max_steps=1 if RUN_MODE == 'smoke' else -1,
        per_device_train_batch_size=MICRO_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type='cosine',
        warmup_steps=5,
        optim='adamw_8bit',
        weight_decay=0.001,
        max_grad_norm=0.3,
        bf16=True,
        tf32=True,
        gradient_checkpointing=True,
        logging_steps=1 if RUN_MODE == 'smoke' else 10,
        report_to='none',
        save_strategy='no',
        eval_strategy='no',
        seed=SEED,
        data_seed=SEED,
        max_length=MAX_SEQ_LENGTH,
        packing=False,
        dataset_num_proc=1,
        run_name='socratic-gemma4-31b-qlora-v2',
    )
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=train_dataset, args=training_args
    )
    trainer = train_on_responses_only(
        trainer, instruction_part='<|turn>user\n', response_part='<|turn>model\n'
    )
    labels = trainer.train_dataset[0]['labels']
    assert any(label == -100 for label in labels), 'Instruction tokens were not masked.'
    assert any(label != -100 for label in labels), 'No assistant tokens remain for loss.'
    print('Response-only loss mask gate: PASS')

Map: 100%|██████████████████████████████████████████████████████████| 400/400 [00:00<00:00, 15698.13 examples/s]

Response-only loss mask gate: PASS


In [10]:
if RUN_MODE == 'inspect':
    print('Inspect mode: optimizer step skipped.')
else:
    started = time.time()
    train_result = trainer.train()
    elapsed_seconds = time.time() - started
    metrics = dict(train_result.metrics)
    assert torch.isfinite(torch.tensor(float(metrics['train_loss'])))
    print('Training returned in minutes:', f'{elapsed_seconds / 60:.1f}')
    print(json.dumps(metrics, indent=2, default=str))
    print('Peak allocated GiB:', f'{torch.cuda.max_memory_allocated() / 2**30:.2f}')
    print('Peak reserved GiB:', f'{torch.cuda.max_memory_reserved() / 2**30:.2f}')
    print('Training loss and memory gate: PASS')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 400 | Num Epochs = 3 | Total steps = 300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 122,429,440 of 31,395,515,952 (0.39% trained)
Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,4.138063
20,0.627590
30,0.170378
40,0.163503
50,0.132151
60,0.121411
70,0.001064
80,0.001632
90,0.000083
100,0.000019


Training returned in minutes: 32.3
{
  "train_runtime": 1935.0843,
  "train_samples_per_second": 0.62,
  "train_steps_per_second": 0.155,
  "total_flos": 4.938408198095674e+16,
  "train_loss": 0.19525247719592395,
  "epoch": 3.0
}
Peak allocated GiB: 19.40
Peak reserved GiB: 19.79
Training loss and memory gate: PASS


## Save the final adapter and evidence

Smoke mode writes only to a temporary directory. Full mode writes the adapter, tokenizer, configuration, metrics, environment report, and hashes needed to reproduce the run.

In [ ]:
def write_manifest(directory: Path, output: Path) -> None:
    files = sorted(path for path in directory.rglob('*') if path.is_file())
    output.write_text(''.join(f'{sha256(path)}  {path.relative_to(ROOT).as_posix()}\n' for path in files))

if RUN_MODE == 'inspect':
    print('Inspect mode: save skipped.')
else:
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    if RUN_MODE == 'smoke':
        print('Temporary smoke adapter saved:', sorted(path.name for path in output_dir.iterdir()))
    else:
        config = {
            'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'load_in_4bit': True, 'max_seq_length': MAX_SEQ_LENGTH},
            'dataset': {'path': str(DATA_PATH.relative_to(ROOT)), 'sha256': sha256(DATA_PATH), 'records': len(rows)},
            'seed': SEED,
            'qlora': {'backend': 'unsloth', 'rank': LORA_R, 'alpha': LORA_ALPHA, 'target_modules': 'all-linear', 'dropout': LORA_DROPOUT},
            'training': {'epochs': EPOCHS, 'learning_rate': LEARNING_RATE, 'scheduler': 'cosine', 'micro_batch_size': MICRO_BATCH_SIZE, 'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS, 'max_length': MAX_SEQ_LENGTH, 'assistant_only_loss': True, 'optimizer': 'adamw_8bit', 'final_checkpoint_only': True},
            'runtime': {'python': os.sys.version.split()[0], 'torch': str(torch.__version__), 'torch_cuda': str(torch.version.cuda) if torch.version.cuda is not None else None, 'gpu': gpu_name, 'gpu_vram_gib': gpu_vram, 'system_ram_gib': total_ram, 'packages': {name: package_version(name) for name in ('unsloth', 'unsloth-zoo', 'bitsandbytes', 'transformers', 'trl', 'datasets', 'accelerate')}},
        }
        import yaml
        FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
        FINAL_LOG_DIR.mkdir(parents=True, exist_ok=True)
        (ROOT / 'train/config-v2.yaml').write_text(yaml.safe_dump(config, sort_keys=False))
        (FINAL_LOG_DIR / 'notebook-log.json').write_text(json.dumps({'metrics': metrics, 'config': config}, indent=2, default=str))
        environment = subprocess.run(['uv', 'pip', 'freeze', '--python', os.sys.executable], text=True, capture_output=True, check=True).stdout
        (FINAL_LOG_DIR / 'environment.txt').write_text(environment)
        write_manifest(FINAL_ADAPTER_DIR, ROOT / 'train/adapter-v2.sha256')
        print('Final adapter and evidence saved.')

## After V2 training

Run the separate 48-case benchmark against the same Gemma 4 base revision and then against the saved adapter. Keep the benchmark prompt, cases, decoding settings, and judge fixed. Compare leakage and actionable diagnosis—not training loss alone.